# IFC4 SampleHouse - COBie Data Transformation

This notebook processes IFC model JSON to add COBie values from an Excel reference table.
- Input: JSON Whole Model/Ifc4_SampleHouse.json and COBie Excel lookup
- Output: JSON_Edit/Ifc4_SampleHouse.json with COBie values added
- Processed Items: IFCWALL, IFCDOOR, IFCWINDOW, IFCSLAB, IFCROOF, IFCFOOTING

In [1]:
from pathlib import Path
from shutil import copy2
import json

import pandas as pd
from IPython.display import display

# ============================================================
# SETUP & PATHS
# ============================================================

workspace_root = Path.cwd().parent
source_json_path = workspace_root / "JSON Whole Model" / "Ifc4_SampleHouse.json"
excel_path = workspace_root / "COBie" / "Uniclass2015_EF_v1_16.xlsx"
json_edit_dir = workspace_root / "JSON_Edit"

assert source_json_path.exists(), f"JSON not found: {source_json_path}"
assert excel_path.exists(), f"Excel not found: {excel_path}"

json_edit_dir.mkdir(parents=True, exist_ok=True)
working_json_path = json_edit_dir / source_json_path.name

if not working_json_path.exists():
    copy2(source_json_path, working_json_path)

print(f"Source JSON: {source_json_path}")
print(f"Working JSON: {working_json_path}")
print(f"Excel Reference: {excel_path}")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_text(value):
    if value is None:
        return ""
    return str(value).strip()


def get_properties(item):
    properties = item.get("Properties", []) if isinstance(item, dict) else []
    return properties if isinstance(properties, list) else []


def get_prop_value(properties, category, display_name):
    category_text = normalize_text(category).upper()
    display_name_text = normalize_text(display_name).upper()

    for prop in properties:
        if not isinstance(prop, dict):
            continue
        prop_category = normalize_text(prop.get("category")).upper()
        prop_display_name = normalize_text(prop.get("displayName")).upper()
        if prop_category == category_text and prop_display_name == display_name_text:
            return normalize_text(prop.get("value"))
    return ""


def is_type_match(item, target_type):
    target = normalize_text(target_type).upper()
    properties = get_properties(item)
    if not target:
        return False

    for prop in properties:
        if not isinstance(prop, dict):
            continue
        category = normalize_text(prop.get("category")).lower()
        display_name = normalize_text(prop.get("displayName")).lower()
        value = normalize_text(prop.get("value")).upper()
        if category == "item" and display_name == "type" and value == target:
            return True
    return False


def extract_items_by_type(data, target_type):
    rows = []
    for item in data:
        if not isinstance(item, dict):
            continue
        if not is_type_match(item, target_type):
            continue

        properties = get_properties(item)
        rows.append(
            {
                "GUID": normalize_text(item.get("ExternalId")),
                "Name": normalize_text(item.get("Name")) or get_prop_value(properties, "Item", "Name"),
                "DbId": item.get("DbId"),
                "Item Type": target_type,
                "Before COBie": get_prop_value(properties, "IFC", "COBie"),
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by=["Name", "GUID"], kind="stable").reset_index(drop=True)
        df.insert(0, "Count", df.index + 1)
    return df


def apply_cobie_value(target_item, cobie_value):
    properties = target_item.get("Properties")
    if not isinstance(properties, list):
        properties = []
        target_item["Properties"] = properties

    existing_cobie_prop = None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if normalize_text(prop.get("category")) == "IFC" and normalize_text(prop.get("displayName")) == "COBie":
            existing_cobie_prop = prop
            break

    before_value = ""
    after_value = normalize_text(cobie_value)

    if existing_cobie_prop is not None:
        before_value = normalize_text(existing_cobie_prop.get("value"))
        if before_value != after_value:
            existing_cobie_prop["value"] = after_value
            return True, before_value, after_value, "Updated"
        return False, before_value, after_value, "Unchanged"

    properties.append({
        "category": "IFC",
        "displayName": "COBie",
        "value": after_value,
    })
    return True, "", after_value, "Added"


def extract_cobie_from_item(item):
    properties = get_properties(item)
    return get_prop_value(properties, "IFC", "COBie")


def build_comparison_table(processed_rows):
    comparison_df = pd.DataFrame(processed_rows)
    if comparison_df.empty:
        return comparison_df

    comparison_df = comparison_df[[
        "Item Type",
        "IFC Type",
        "Total Items",
        "Name of the model element",
        "COBie Value",
        "Change",
    ]].copy()

    comparison_df = comparison_df.sort_values(
        by=["Item Type", "IFC Type", "Name of the model element"],
        kind="stable",
    ).reset_index(drop=True)

    return comparison_df


# ============================================================
# LOAD DATA & SEARCH EXCEL FOR COBIE MAPPINGS
# ============================================================

with source_json_path.open("r", encoding="utf-8") as f:
    source_data = json.load(f)

with working_json_path.open("r", encoding="utf-8") as f:
    working_data = json.load(f)

ef_df = pd.read_excel(excel_path, sheet_name="EF", header=2)

items_to_process = [
    {"ifc_type": "IFCWALL", "excel_search": "wall", "display_name": "Walls"},
    {"ifc_type": "IFCDOOR", "excel_search": "door", "display_name": "Doors"},
    {"ifc_type": "IFCWINDOW", "excel_search": "window", "display_name": "Windows"},
    {"ifc_type": "IFCSLAB", "excel_search": "roofs, floor and paving elements", "display_name": "Slabs"},
    {"ifc_type": "IFCROOF", "excel_search": "roof", "display_name": "Roofs"},
    {"ifc_type": "IFCFOOTING", "excel_search": "foundation", "display_name": "Footings"},
]

cobie_mapping = {}
for config in items_to_process:
    ifc_type = config["ifc_type"]
    search_term = config["excel_search"]
    display_name = config["display_name"]

    matched_rows = ef_df[ef_df["Title"].astype(str).str.contains(search_term, case=False, na=False)].copy()
    matched_rows = matched_rows.sort_values(by=["Title", "Code"], kind="stable").reset_index(drop=True)

    if not matched_rows.empty:
        cobie_value = normalize_text(matched_rows.iloc[0]["COBie"])
        cobie_mapping[ifc_type] = {
            "value": cobie_value,
            "excel_title_df": matched_rows,
            "count": len(matched_rows),
            "display_name": display_name,
        }
        print(f"{display_name:10s} - COBie value found: {cobie_value}")
    else:
        print(f"{display_name:10s} - WARNING: No COBie mapping found in Excel!")

print(f"\nTotal mapped IFC types: {len(cobie_mapping)}")

# ============================================================
# APPLY COBIE VALUES AND PRINT COMPARISON TABLE
# ============================================================

working_by_guid = {}
for item in working_data:
    if not isinstance(item, dict):
        continue
    guid = normalize_text(item.get("ExternalId"))
    if guid:
        working_by_guid[guid] = item

processed_rows = []
processing_summary_rows = []

for config in items_to_process:
    ifc_type = config["ifc_type"]
    display_name = config["display_name"]
    mapping = cobie_mapping.get(ifc_type)

    source_df = extract_items_by_type(source_data, ifc_type)
    total_items = len(source_df)

    updated_count = 0
    added_count = 0
    unchanged_count = 0
    missing_guid_count = 0

    cobie_value = mapping["value"] if mapping else ""

    if source_df.empty:
        processing_summary_rows.append(
            {
                "Item Type": display_name,
                "IFC Type": ifc_type,
                "Total Items": 0,
                "Updated": 0,
                "Added": 0,
                "Unchanged": 0,
                "Missing GUIDs": 0,
                "COBie Value": cobie_value,
            }
        )
        continue

    for row in source_df.itertuples(index=False):
        target_item = working_by_guid.get(normalize_text(row.GUID))
        before_value = normalize_text(getattr(row, "Before COBie", ""))
        after_value = before_value
        change = "Skipped"

        if not isinstance(target_item, dict):
            missing_guid_count += 1
            change = "Missing GUID"
        elif not cobie_value:
            change = "No Mapping"
        else:
            was_updated, _, after_value, change = apply_cobie_value(target_item, cobie_value)
            if change == "Added":
                added_count += 1
            elif change == "Updated":
                updated_count += 1
            elif change == "Unchanged":
                unchanged_count += 1
            if not was_updated and change not in {"Unchanged"}:
                unchanged_count += 1

        name_of_element = normalize_text(row.Name)
        processed_rows.append(
            {
                "Item Type": display_name,
                "IFC Type": ifc_type,
                "Total Items": total_items,
                "Name of the model element": name_of_element,
                "COBie Value": after_value or cobie_value,
                "Change": change,
            }
        )

    processing_summary_rows.append(
        {
            "Item Type": display_name,
            "IFC Type": ifc_type,
            "Total Items": total_items,
            "Updated": updated_count,
            "Added": added_count,
            "Unchanged": unchanged_count,
            "Missing GUIDs": missing_guid_count,
            "COBie Value": cobie_value,
        }
    )

with working_json_path.open("w", encoding="utf-8") as f:
    json.dump(working_data, f, ensure_ascii=False, indent=2)

comparison_table_df = build_comparison_table(processed_rows)
processing_summary_df = pd.DataFrame(processing_summary_rows)

print("=" * 80)
print("TABLE COMPARISON")
print("=" * 80)
display(comparison_table_df)

print("\n" + "=" * 80)
print("PROCESSING SUMMARY")
print("=" * 80)
display(processing_summary_df)

print(f"\nOutput Path: {working_json_path}")

Source JSON: c:\Git\APS-IFC\JSON Whole Model\Ifc4_SampleHouse.json
Working JSON: c:\Git\APS-IFC\JSON_Edit\Ifc4_SampleHouse.json
Excel Reference: c:\Git\APS-IFC\COBie\Uniclass2015_EF_v1_16.xlsx
Walls      - COBie value found: EF_25_10_25 : External walls
Doors      - COBie value found: EF_25_30_25 : Doors
Windows    - COBie value found: EF_25_30_97 : Windows
Slabs      - COBie value found: EF_30 : Roofs, floor and paving elements
Roofs      - COBie value found: EF_30_10_04 : Arched roofs
Footings   - COBie value found: EF_20_05_30 : Foundations

Total mapped IFC types: 6
TABLE COMPARISON


,Item Type,IFC Type,Total Items,Name of the model element,COBie Value,Change
0,Doors,IFCDOOR,3,Doors_ExtDbl_Flush:1810x2110mm:285860,EF_25_30_25 : Doors,Added
1,Doors,IFCDOOR,3,Doors_IntSgl:810x2110mm:285959,EF_25_30_25 : Doors,Added
2,Doors,IFCDOOR,3,Doors_IntSgl:810x2110mm:285996,EF_25_30_25 : Doors,Added
3,Roofs,IFCROOF,1,Basic Roof:Roof_Flat-4Felt-150Ins-50Scr-150Con...,EF_30_10_04 : Arched roofs,Added
4,Slabs,IFCSLAB,2,Floor:Floor-Grnd-Susp_65Scr-80Ins-100Blk-75PC:...,"EF_30 : Roofs, floor and paving elements",Added
5,Slabs,IFCSLAB,2,Floor:Simple floor:295048,"EF_30 : Roofs, floor and paving elements",Added
6,Walls,IFCWALL,3,Basic Wall:Wall-Ext_102Bwk-75Ins-100LBlk-12P:2...,EF_25_10_25 : External walls,Added
7,Walls,IFCWALL,3,Basic Wall:Wall-Ext_102Bwk-75Ins-100LBlk-12P:2...,EF_25_10_25 : External walls,Added
8,Walls,IFCWALL,3,Basic Wall:Wall-Ext_102Bwk-75Ins-100LBlk-12P:2...,EF_25_10_25 : External walls,Added
9,Windows,IFCWINDOW,4,Windows_Sgl_Plain:1810x1210mm:286105,EF_25_30_97 : Windows,Added



PROCESSING SUMMARY


,Item Type,IFC Type,Total Items,Updated,Added,Unchanged,Missing GUIDs,COBie Value
0,Walls,IFCWALL,3,0,3,0,0,EF_25_10_25 : External walls
1,Doors,IFCDOOR,3,0,3,0,0,EF_25_30_25 : Doors
2,Windows,IFCWINDOW,4,0,4,0,0,EF_25_30_97 : Windows
3,Slabs,IFCSLAB,2,0,2,0,0,"EF_30 : Roofs, floor and paving elements"
4,Roofs,IFCROOF,1,0,1,0,0,EF_30_10_04 : Arched roofs
5,Footings,IFCFOOTING,0,0,0,0,0,EF_20_05_30 : Foundations



Output Path: c:\Git\APS-IFC\JSON_Edit\Ifc4_SampleHouse.json
